In [0]:
print("Ecommerce Retail Analytics Project")

print("Starting PySpark on Databricks!")

Ecommerce Retail Analytics Project
Starting PySpark on Databricks!


In [0]:
print("Spark version:", spark.version)

display(spark.range(5))

Spark version: 4.2.0


id
0
1
2
3
4


In [0]:
print("Spark version:", spark.version)

Spark version: 4.2.0


In [0]:
file_path = "/Workspace/Users/surbhiaglawe143@gmail.com/online_retail.csv"

sales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .csv(file_path)
)

print("Total rows:", sales_df.count())
print("Total columns:", len(sales_df.columns))

sales_df.printSchema()
display(sales_df.limit(10))

Total rows: 541909
Total columns: 8
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,2.55,17850.0,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01T08:26:00.000Z,3.39,17850.0,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01T08:26:00.000Z,2.75,17850.0,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01T08:26:00.000Z,3.39,17850.0,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00.000Z,3.39,17850.0,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01T08:26:00.000Z,7.65,17850.0,United Kingdom
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,4.25,17850.0,United Kingdom
536366,22633,HAND WARMER UNION JACK,6,2010-12-01T08:28:00.000Z,1.85,17850.0,United Kingdom
536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01T08:28:00.000Z,1.85,17850.0,United Kingdom
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01T08:34:00.000Z,1.69,13047.0,United Kingdom


In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    when,
    count,
    sum as spark_sum
)

print("========== DATA QUALITY REPORT ==========")

print("Total rows:", sales_df.count())

print("Duplicate rows:", 
      sales_df.count() - sales_df.dropDuplicates().count())

print("\nMissing values by column:")

sales_df.select([
    spark_sum(
        when(
            col(c).isNull() |
            (trim(col(c)) == "" if c in ["Description", "CustomerID", "Country"] else col(c).isNull()),
            1
        ).otherwise(0)
    ).alias(c)
    for c in sales_df.columns
]).show(truncate=False)

========== DATA QUALITY REPORT ==========
Total rows: 541909
Duplicate rows: 5268

Missing values by column:
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|0        |0        |1454       |0       |0          |0        |135080    |0      |
+---------+---------+-----------+--------+-----------+---------+----------+-------+



In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    when,
    lit,
    to_timestamp,
    round as spark_round
)

# 1. Remove exact duplicate rows
clean_df = sales_df.dropDuplicates()

# 2. Clean text columns
for column_name in ["Description", "CustomerID", "Country"]:
    clean_df = clean_df.withColumn(
        column_name,
        when(
            col(column_name).isNull() |
            (trim(col(column_name)) == ""),
            lit(None).cast("string")
        ).otherwise(trim(col(column_name)))
    )

# 3. Add data-quality flags
clean_df = clean_df.withColumn(
    "MissingDescription",
    col("Description").isNull()
)

clean_df = clean_df.withColumn(
    "MissingCustomerID",
    col("CustomerID").isNull()
)

clean_df = clean_df.withColumn(
    "NegativeQuantity",
    col("Quantity") < 0
)

clean_df = clean_df.withColumn(
    "InvalidPrice",
    col("UnitPrice").isNull() | (col("UnitPrice") < 0)
)

clean_df = clean_df.withColumn(
    "ZeroPrice",
    col("UnitPrice") == 0
)

# 4. Calculate revenue
clean_df = clean_df.withColumn(
    "Revenue",
    when(
        col("Quantity").isNotNull() &
        col("UnitPrice").isNotNull(),
        spark_round(col("Quantity") * col("UnitPrice"), 2)
    ).otherwise(lit(None).cast("double"))
)

# 5. Verify the result
print("Original rows:", sales_df.count())
print("Cleaned rows:", clean_df.count())
print("Rows removed:", sales_df.count() - clean_df.count())

display(clean_df.limit(10))

Original rows: 541909
Cleaned rows: 536641
Rows removed: 5268


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,MissingDescription,MissingCustomerID,NegativeQuantity,InvalidPrice,ZeroPrice,Revenue
536367,22748,POPPY'S PLAYHOUSE KITCHEN,6,2010-12-01T08:34:00.000Z,2.1,13047.0,United Kingdom,false,false,false,false,false,12.6
536390,21533,RETROSPOT LARGE MILK JUG,12,2010-12-01T10:19:00.000Z,4.25,17511.0,United Kingdom,false,false,false,false,false,51.0
536395,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01T10:47:00.000Z,1.69,13767.0,United Kingdom,false,false,false,false,false,54.08
536395,22114,HOT WATER BOTTLE TEA AND SYMPATHY,8,2010-12-01T10:47:00.000Z,3.95,13767.0,United Kingdom,false,false,false,false,false,31.6
536396,21071,VINTAGE BILLBOARD DRINK ME MUG,6,2010-12-01T10:51:00.000Z,1.06,17850.0,United Kingdom,false,false,false,false,false,6.36
536401,21464,DISCO BALL ROTATOR BATTERY OPERATED,1,2010-12-01T11:21:00.000Z,4.25,15862.0,United Kingdom,false,false,false,false,false,4.25
536401,22068,BLACK PIRATE TREASURE CHEST,2,2010-12-01T11:21:00.000Z,1.65,15862.0,United Kingdom,false,false,false,false,false,3.3
536404,21062,PARTY INVITES SPACEMAN,12,2010-12-01T11:29:00.000Z,0.85,16218.0,United Kingdom,false,false,false,false,false,10.2
536406,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01T11:33:00.000Z,7.65,17850.0,United Kingdom,false,false,false,false,false,15.3
536408,22371,AIRLINE BAG VINTAGE TOKYO 78,4,2010-12-01T11:41:00.000Z,4.25,14307.0,United Kingdom,false,false,false,false,false,17.0


In [0]:
# Save the cleaned DataFrame as a managed Delta table
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("cleaned_online_retail")

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [0]:
saved_df = spark.table("cleaned_online_retail")

print("Saved row count:", saved_df.count())

display(saved_df.limit(10))

Saved row count: 536641


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,MissingDescription,MissingCustomerID,NegativeQuantity,InvalidPrice,ZeroPrice,Revenue
536367,22748,POPPY'S PLAYHOUSE KITCHEN,6,2010-12-01T08:34:00.000Z,2.1,13047.0,United Kingdom,false,false,false,false,false,12.6
536390,21533,RETROSPOT LARGE MILK JUG,12,2010-12-01T10:19:00.000Z,4.25,17511.0,United Kingdom,false,false,false,false,false,51.0
536395,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01T10:47:00.000Z,1.69,13767.0,United Kingdom,false,false,false,false,false,54.08
536395,22114,HOT WATER BOTTLE TEA AND SYMPATHY,8,2010-12-01T10:47:00.000Z,3.95,13767.0,United Kingdom,false,false,false,false,false,31.6
536396,21071,VINTAGE BILLBOARD DRINK ME MUG,6,2010-12-01T10:51:00.000Z,1.06,17850.0,United Kingdom,false,false,false,false,false,6.36
536401,21464,DISCO BALL ROTATOR BATTERY OPERATED,1,2010-12-01T11:21:00.000Z,4.25,15862.0,United Kingdom,false,false,false,false,false,4.25
536401,22068,BLACK PIRATE TREASURE CHEST,2,2010-12-01T11:21:00.000Z,1.65,15862.0,United Kingdom,false,false,false,false,false,3.3
536404,21062,PARTY INVITES SPACEMAN,12,2010-12-01T11:29:00.000Z,0.85,16218.0,United Kingdom,false,false,false,false,false,10.2
536406,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01T11:33:00.000Z,7.65,17850.0,United Kingdom,false,false,false,false,false,15.3
536408,22371,AIRLINE BAG VINTAGE TOKYO 78,4,2010-12-01T11:41:00.000Z,4.25,14307.0,United Kingdom,false,false,false,false,false,17.0


In [0]:
from pyspark.sql.functions import sum as spark_sum, round as spark_round, col

# Load our saved, cleaned Delta table
saved_df = spark.table("cleaned_online_retail")

# Calculate revenue by country
country_sales = (
    saved_df
    .groupBy("Country")
    .agg(
        spark_round(spark_sum("Revenue"), 2).alias("TotalRevenue")
    )
    .orderBy(col("TotalRevenue").desc())
)

display(country_sales.limit(10))

Country,TotalRevenue
United Kingdom,8167128.18
Netherlands,284661.54
EIRE,262993.38
Germany,221509.47
France,197317.11
Australia,137009.77
Switzerland,56363.05
Spain,54756.03
Belgium,40910.96
Sweden,36585.41


In [0]:
from pyspark.sql.functions import sum as spark_sum, round as spark_round, col

product_sales = (
    saved_df
    .groupBy("StockCode", "Description")
    .agg(
        spark_round(
            spark_sum("Revenue"), 2
        ).alias("TotalRevenue"),
        spark_sum("Quantity").alias("TotalQuantitySold")
    )
    .orderBy(col("TotalRevenue").desc())
)

display(product_sales.limit(10))

StockCode,Description,TotalRevenue,TotalQuantitySold
DOT,DOTCOM POSTAGE,206245.48,707
22423,REGENCY CAKESTAND 3 TIER,164459.49,13007
47566,PARTY BUNTING,98243.88,18010
85123A,WHITE HANGING HEART T-LIGHT HOLDER,97659.94,35006
85099B,JUMBO BAG RED RETROSPOT,92175.79,47260
23084,RABBIT NIGHT LIGHT,66661.63,30631
POST,POSTAGE,66230.64,3003
22086,PAPER CHAIN KIT 50'S CHRISTMAS,63715.24,18876
84879,ASSORTED COLOUR BIRD ORNAMENT,58792.42,36282
79321,CHILLI LIGHTS,53746.66,10225


In [0]:
from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    round as spark_round,
    date_format
)

monthly_sales = (
    saved_df
    .withColumn(
        "SalesMonth",
        date_format(col("InvoiceDate"), "yyyy-MM")
    )
    .groupBy("SalesMonth")
    .agg(
        spark_round(
            spark_sum("Revenue"), 2
        ).alias("TotalRevenue")
    )
    .orderBy("SalesMonth")
)

display(monthly_sales)

SalesMonth,TotalRevenue
2010-12,746723.61
2011-01,558448.56
2011-02,497026.41
2011-03,682013.98
2011-04,492367.84
2011-05,722094.1
2011-06,689977.23
2011-07,680156.99
2011-08,681386.46
2011-09,1017596.68


In [0]:
from pyspark.sql.functions import (
    sum as spark_sum,
    round as spark_round,
    countDistinct
)

business_summary = saved_df.agg(
    spark_round(
        spark_sum("Revenue"), 2
    ).alias("TotalRevenue"),

    spark_sum("Quantity").alias("NetQuantity"),

    countDistinct("InvoiceNo").alias("UniqueInvoices"),

    countDistinct("CustomerID").alias("UniqueCustomers")
)

display(business_summary)

TotalRevenue,NetQuantity,UniqueInvoices,UniqueCustomers
9726006.95,5162502,25900,4372


In [0]:
saved_df.createOrReplaceTempView("retail_sales")

print("Temporary SQL view created!")

Temporary SQL view created!


In [0]:
%sql
SELECT
    Country,
    ROUND(SUM(Revenue), 2) AS TotalRevenue
FROM retail_sales
GROUP BY Country
ORDER BY TotalRevenue DESC
LIMIT 10;

Country,TotalRevenue
United Kingdom,8167128.18
Netherlands,284661.54
EIRE,262993.38
Germany,221509.47
France,197317.11
Australia,137009.77
Switzerland,56363.05
Spain,54756.03
Belgium,40910.96
Sweden,36585.41


In [0]:
%sql
SELECT
    StockCode,
    Description,
    ROUND(SUM(Revenue), 2) AS TotalRevenue
FROM retail_sales
WHERE Revenue IS NOT NULL
GROUP BY StockCode, Description
HAVING SUM(Revenue) > 50000
ORDER BY TotalRevenue DESC;

StockCode,Description,TotalRevenue
DOT,DOTCOM POSTAGE,206245.48
22423,REGENCY CAKESTAND 3 TIER,164459.49
47566,PARTY BUNTING,98243.88
85123A,WHITE HANGING HEART T-LIGHT HOLDER,97659.94
85099B,JUMBO BAG RED RETROSPOT,92175.79
23084,RABBIT NIGHT LIGHT,66661.63
POST,POSTAGE,66230.64
22086,PAPER CHAIN KIT 50'S CHRISTMAS,63715.24
84879,ASSORTED COLOUR BIRD ORNAMENT,58792.42
79321,CHILLI LIGHTS,53746.66


In [0]:
from pyspark.sql.functions import col

print("========== DATA QUALITY VALIDATION ==========")

# Check 1: Ensure no exact duplicate rows remain
duplicate_count = (
    saved_df.count() - saved_df.dropDuplicates().count()
)
print("Remaining duplicate rows:", duplicate_count)

# Check 2: Check for missing invoice numbers
missing_invoices = saved_df.filter(
    col("InvoiceNo").isNull()
).count()
print("Missing invoice numbers:", missing_invoices)

# Check 3: Check for missing dates
missing_dates = saved_df.filter(
    col("InvoiceDate").isNull()
).count()
print("Missing invoice dates:", missing_dates)

# Check 4: Check the revenue calculation
incorrect_revenue = saved_df.filter(
    col("Revenue").isNotNull()
    & col("Quantity").isNotNull()
    & col("UnitPrice").isNotNull()
    & (
        col("Revenue")
        != __import__("pyspark.sql.functions", fromlist=["round"])
        .round(col("Quantity") * col("UnitPrice"), 2)
    )
).count()

print("Incorrect revenue rows:", incorrect_revenue)

# Check 5: Verify expected row count
actual_rows = saved_df.count()
expected_rows = 536641

print("Actual rows:", actual_rows)
print("Expected rows:", expected_rows)
print("Row count matches:", actual_rows == expected_rows)

========== DATA QUALITY VALIDATION ==========
Remaining duplicate rows: 0
Missing invoice numbers: 0
Missing invoice dates: 0
Incorrect revenue rows: 0
Actual rows: 536641
Expected rows: 536641
Row count matches: True


In [0]:
from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    round as spark_round
)

merchandise_df = saved_df.filter(
    ~col("StockCode").isin("DOT", "POST")
)

merchandise_sales = (
    merchandise_df
    .groupBy("StockCode", "Description")
    .agg(
        spark_round(
            spark_sum("Revenue"), 2
        ).alias("TotalRevenue"),
        spark_sum("Quantity").alias("NetQuantity")
    )
    .orderBy(col("TotalRevenue").desc())
)

display(merchandise_sales.limit(10))

StockCode,Description,TotalRevenue,NetQuantity
22423,REGENCY CAKESTAND 3 TIER,164459.49,13007
47566,PARTY BUNTING,98243.88,18010
85123A,WHITE HANGING HEART T-LIGHT HOLDER,97659.94,35006
85099B,JUMBO BAG RED RETROSPOT,92175.79,47260
23084,RABBIT NIGHT LIGHT,66661.63,30631
22086,PAPER CHAIN KIT 50'S CHRISTMAS,63715.24,18876
84879,ASSORTED COLOUR BIRD ORNAMENT,58792.42,36282
79321,CHILLI LIGHTS,53746.66,10225
23298,SPOTTY BUNTING,42030.67,8381
22386,JUMBO BAG PINK POLKADOT,41584.43,20992


In [0]:
%sql
CREATE OR REPLACE TABLE monthly_sales_summary AS
SELECT
    DATE_FORMAT(InvoiceDate, 'yyyy-MM') AS SalesMonth,
    ROUND(SUM(Revenue), 2) AS TotalRevenue
FROM retail_sales
GROUP BY DATE_FORMAT(InvoiceDate, 'yyyy-MM')
ORDER BY SalesMonth;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM monthly_sales_summary
ORDER BY SalesMonth;

SalesMonth,TotalRevenue
2010-12,746723.61
2011-01,558448.56
2011-02,497026.41
2011-03,682013.98
2011-04,492367.84
2011-05,722094.1
2011-06,689977.23
2011-07,680156.99
2011-08,681386.46
2011-09,1017596.68
